# SPARQL-Relax Tutorial

This notebook demonstrates how to use the `sparql-relax` Python bindings to diagnose and repair broken SPARQL queries.

## What is SPARQL-Relax?
SPARQL-Relax helps find why a SPARQL query returns no results when it's expected to, and attempts to find a "connected" version of the query that does return results by searching for paths in the knowledge graph.

In [ ]:
from sparql_relax import diagnose, diagnose_and_connect, query, Store
import os

## 1. Setup Data

First, we load our knowledge graph (TTL file). We'll use the `b59` building dataset.

In [ ]:
data_path = "/home/lazlo/Desktop/semantics/sparql-relax/eval/buildings/b59.ttl"
with open(data_path, "r") as f:
    data = f.read()
print(f"Loaded data from {data_path}")

## 2. Running a Correct Query

Let's start with a query that we know works.

In [ ]:
correct_query = """
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX ex1: <http://data.ashrae.org/standard223/data/lbnl-example-2#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX ref: <https://brickschema.org/schema/Brick/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

SELECT DISTINCT ?target ?label ?alc_endpoint ?mpc_endpoint WHERE {
{ ?target a s223:Zone; s223:hasProperty ?prop. }
UNION
{ ?target a s223:Zone; s223:hasDomainSpace ?space .
?space a ex1:RoomDomainSpace ;
s223:connected/s223:connected/s223:contains/s223:hasProperty ?prop . }

?prop a s223:Property;
qudt:hasQuantityKind|s223:hasEnumerationKind ?quantKind;
rdfs:label ?label;
s223:hasExternalReference ?ref .

?ref ref:endpoint ?alc_endpoint;
ref:mpc_db_name ?mpc_endpoint;
ref:writable "False" .
}
"""

results = query(data, correct_query)
print(f"Found {len(results.bindings)} results.")
for row in results.bindings[:5]:
    print(row)


## 3. Diagnosing a Broken Query

Now, let's take a query that is slightly broken (e.g., uses a wrong predicate). This query should return no results.

In [ ]:
broken_query = """
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX s223: <http://data.ashrae.org/standard223#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?zone ?propLabel ?alcEndpoint ?mpcEndpoint
WHERE {
  ?zone a s223:Zone .
  {
    ?zone sh:property ?propLabel .
    FILTER(?propLabel = "node65")
  } UNION {
    ?zone sh:property ?propLabel .
    FILTER(?propLabel = "node66")
  }
  BIND(brick:Unoccupied_Zone_Air_Temperature_Setpoint AS ?alcEndpoint)
  BIND(brick:Zone_Air_Heating_Temperature_Setpoint AS ?mpcEndpoint)
}
"""

results = query(data, broken_query)
print(f"Found {len(results.bindings)} results (expected 0).")

print("Diagnosing...")
diagnosis = diagnose(data, broken_query)
for culprit in diagnosis.culprits:
    print(f"Culprit found: {culprit.triple}")

## 4. Connecting the Query

Now we use `diagnose_and_connect` to automatically find a fix for the broken query.

In [ ]:
print("Connecting query...")
report = diagnose_and_connect(data, broken_query)

for result in report.results:
    if result.fixed:
        print(f"Fixed triple: {result.triple} -> {result.path_text}")
        print("Connected Query:\n", result.connected_query)
        
        # Verify the fixed query
        fixed_results = query(data, result.connected_query)
        print(f"Fixed query found {len(fixed_results.bindings)} results.\n")

## 5. Efficient Usage with Store

If you are running multiple queries against the same data, use a `Store` to avoid reparsing the TTL file every time.

In [ ]:
store = Store(data)
results = store.query(correct_query)
print(f"Store query found {len(results.bindings)} results.")